In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
%autosave 300

In [ ]:
import os

os.chdir("../")
print(os.getcwd())

#### Multiclass XOR Classification Example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchmetrics
import torchvision.utils as vutils
from PIL import Image
from torchvision import transforms
import torchvision

In [ ]:
data = pd.read_csv("data/xor.csv")
data

In [ ]:
data["class label"].value_counts()

In [ ]:
X = data[["x1", "x2"]].values
y = data["class label"].values

print(X.shape, y.shape)

In [ ]:
# split into train, validation and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)
print(np.bincount(y_train), np.bincount(y_val), np.bincount(y_test))

In [ ]:
plt.plot(
    X_train[y_train == 0, 0],
    X_train[y_train == 0, 1],
    marker="D",
    markersize=10,
    linestyle="",
    label="Class 0",
)

plt.plot(
    X_train[y_train == 1, 0],
    X_train[y_train == 1, 1],
    marker="^",
    markersize=13,
    linestyle="",
    label="Class 1",
)

plt.legend(loc=2)

plt.xlim([-5, 5])
plt.ylim([-5, 5])

plt.xlabel("Feature $x_1$", fontsize=12)
plt.ylabel("Feature $x_2$", fontsize=12)

plt.grid()
plt.show()

In [ ]:
class XorModel(nn.Module):
    def __init__(self, in_features, hidden_units, out_features):
        super(XorModel, self).__init__()
        self.layer_1 = nn.Linear(in_features, hidden_units)
        self.layer_2 = nn.Linear(hidden_units, 15)
        self.layer_3 = nn.Linear(15, out_features)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.relu(x)
        x = self.layer_3(x)
        return x

In [ ]:
class XORDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __getitem__(self, index):
        x = self.X[index]
        y = self.y[index]
        return x, y

    def __len__(self):
        return self.X.shape[0]

In [ ]:
train_ds = XORDataset(X_train, y_train)
val_ds = XORDataset(X_val, y_val)
test_ds = XORDataset(X_test, y_test)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=32,
    shuffle=True,
)

val_loader = DataLoader(
    dataset=val_ds,
    batch_size=32,
    shuffle=False,
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=32,
    shuffle=False,
)

In [ ]:
# for X_batch, y_batch in train_loader:
#     print(X_batch.shape, y_batch.shape)
#     break

# for X_batch, y_batch in val_loader:
#     print(X_batch.shape, y_batch.shape)
#     break

# for X_batch, y_batch in test_loader:
#     print(X_batch.shape, y_batch.shape)
#     break

In [ ]:
metric = torchmetrics.Accuracy(task="multiclass", num_classes=2)

In [ ]:
def training_loop(model, train_loader, criterion, optimizer, num_epochs, metric):
    """Training loop for the model."""
    for epoch in range(num_epochs):
        model = model.train()
        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            # Forward pass
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

        # calculate metrics
        accuracy = metric.compute()
        metric.reset()

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}")

In [ ]:
def evaluate_model(model, test_loader, criterion, metric):
    """Evaluate the model on the test set."""
    model = model.eval()
    test_loss = 0.0
    with torch.inference_mode():
        for X_batch, y_batch in test_loader:
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            test_loss += loss.item()
            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

    accuracy = metric.compute()
    metric.reset()
    test_loss /= len(test_loader)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {accuracy:.4f}")

In [ ]:
model = XorModel(in_features=2, hidden_units=8, out_features=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

In [ ]:
training_loop(model, train_loader, criterion, optimizer, num_epochs=200, metric=metric)

In [ ]:
evaluate_model(model, train_loader, criterion, metric)
evaluate_model(model, val_loader, criterion, metric)
evaluate_model(model, test_loader, criterion, metric)

In [ ]:
from matplotlib.colors import ListedColormap
import numpy as np


def plot_decision_regions(X, y, classifier, resolution=0.02):

    # setup marker generator and color map
    markers = ("D", "^", "x", "s", "v")
    colors = ("C0", "C1", "C2", "C3", "C4")
    cmap = ListedColormap(colors[: len(np.unique(y))])

    # plot the decision surface
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, resolution), np.arange(x2_min, x2_max, resolution))
    tensor = torch.tensor(np.array([xx1.ravel(), xx2.ravel()]).T).float()
    logits = classifier.forward(tensor)
    Z = np.argmax(logits.detach().numpy(), axis=1)

    Z = Z.reshape(xx1.shape)
    plt.contourf(xx1, xx2, Z, alpha=0.4, cmap=cmap)
    plt.xlim(xx1.min(), xx1.max())
    plt.ylim(xx2.min(), xx2.max())

    # plot class samples
    for idx, cl in enumerate(np.unique(y)):
        plt.scatter(
            x=X[y == cl, 0],
            y=X[y == cl, 1],
            alpha=0.8,
            color=cmap(idx),
            # edgecolor='black',
            marker=markers[idx],
            label=cl,
        )

In [ ]:
plot_decision_regions(X_train, y_train, classifier=model)

##### MNIST Classification Example

In [ ]:
import os
from git import Repo

if not os.path.exists("mnist-pngs"):
    Repo.clone_from("https://github.com/rasbt/mnist-pngs", "mnist-pngs")

In [ ]:
df_train = pd.read_csv("mnist-pngs/train.csv")
df_train.head()

In [ ]:
df_test = pd.read_csv("mnist-pngs/test.csv")
df_test.head()

In [ ]:
df_train = pd.read_csv("mnist-pngs/train.csv")
df_train = df_train.sample(frac=1, random_state=123)

loc = round(df_train.shape[0] * 0.9)
df_new_train = df_train.iloc[:loc]
df_new_val = df_train.iloc[loc:]

df_new_train.to_csv("mnist-pngs/new_train.csv", index=None)
df_new_val.to_csv("mnist-pngs/new_val.csv", index=None)

In [ ]:
class MnistDataset(Dataset):
    """
    Custom Dataset for loading MNIST images and labels from a CSV file.
    The CSV file should contain two columns: 'filepath' and 'label'.
    'filepath' contains the relative path to the image file.
    'label' contains the corresponding class label for the image.
    """

    def __init__(self, csv_path, img_dir, transform=None):
        df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

        self.img_names = df["filepath"]
        self.labels = df["label"]

    def __getitem__(self, index):
        img = Image.open(os.path.join(self.img_dir, self.img_names[index]))

        if self.transform is not None:
            img = self.transform(img)

        label = self.labels[index]
        return img, label

    def __len__(self):
        return self.labels.shape[0]

In [ ]:
def viz_batch_images(batch):

    plt.figure(figsize=(8, 8))
    plt.axis("off")
    plt.title("Training images")
    plt.imshow(np.transpose(vutils.make_grid(batch[0][:64], padding=2, normalize=True), (1, 2, 0)))
    plt.show()

In [ ]:
data_transforms = {
    "train": transforms.Compose(
        [
            transforms.Resize(32),
            transforms.CenterCrop((28, 28)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ]
    ),
    "test": transforms.Compose(
        [
            transforms.Resize(32),
            transforms.CenterCrop((28, 28)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ]
    ),
}

In [ ]:
train_dataset = MnistDataset(
    csv_path="mnist-pngs/new_train.csv", img_dir="mnist-pngs", transform=data_transforms["train"]
)

val_dataset = MnistDataset(csv_path="mnist-pngs/new_val.csv", img_dir="mnist-pngs", transform=data_transforms["test"])

test_dataset = MnistDataset(csv_path="mnist-pngs/test.csv", img_dir="mnist-pngs", transform=data_transforms["test"])

In [ ]:
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
)

In [ ]:
batch = next(iter(train_dataloader))
viz_batch_images(batch[0])

In [ ]:
for X_batch, y_batch in train_dataloader:
    print(X_batch.shape, y_batch.shape)
    break

# for X_batch, y_batch in val_dataloader:
#     print(X_batch.shape, y_batch.shape)
#     break

# for X_batch, y_batch in test_dataloader:
#     print(X_batch.shape, y_batch.shape)
#     break

In [ ]:
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Training images")
plt.imshow(np.transpose(torchvision.utils.make_grid(X_batch[:64], padding=1, pad_value=1.0, normalize=True), (1, 2, 0)))
plt.show()

In [ ]:
class MnistModel(nn.Module):
    """
    Simple Feedforward Neural Network for MNIST classification.
    """

    def __init__(self, in_features, hidden_units, out_features):
        super(MnistModel, self).__init__()
        self.layer_1 = nn.Linear(in_features, hidden_units)
        self.layer_2 = nn.Linear(hidden_units, 64)
        self.layer_3 = nn.Linear(64, out_features)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.relu(x)
        x = self.layer_3(x)
        return x

In [ ]:
metric = torchmetrics.Accuracy(task="multiclass", num_classes=10)

In [ ]:
model = MnistModel(in_features=28 * 28, hidden_units=128, out_features=10)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [ ]:
training_loop(model, train_dataloader, criterion, optimizer, num_epochs=10, metric=metric)

In [ ]:
evaluate_model(model, train_dataloader, criterion, metric)
evaluate_model(model, val_dataloader, criterion, metric)
evaluate_model(model, test_dataloader, criterion, metric)

#### Using MPS backend for Mac Users

In [ ]:
print(f"torch backend MPS is available? {torch.backends.mps.is_available()}")
print(f"current PyTorch installation built with MPS activated? {torch.backends.mps.is_built()}")

In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
def training_loop_device(model, train_loader, criterion, optimizer, num_epochs, metric, device):
    """Training loop for the model."""
    metric = metric.to(device)
    for epoch in range(num_epochs):
        model = model.train()
        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            # Forward pass
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

        # calculate metrics
        accuracy = metric.compute()
        metric.reset()

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}")

In [ ]:
def evaluate_model_device(model, test_loader, criterion, metric, device):
    """Evaluate the model on the test set."""
    metric = metric.to(device)
    model = model.eval()
    test_loss = 0.0
    with torch.inference_mode():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            test_loss += loss.item()
            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

    accuracy = metric.compute()
    metric.reset()
    test_loss /= len(test_loader)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {accuracy:.4f}")

In [ ]:
metric = torchmetrics.Accuracy(task="multiclass", num_classes=10)
model = MnistModel(in_features=28 * 28, hidden_units=128, out_features=10)
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [ ]:
training_loop_device(model, train_dataloader, criterion, optimizer, num_epochs=10, metric=metric, device=device)

In [ ]:
evaluate_model_device(model, train_dataloader, criterion, metric, device=device)
evaluate_model_device(model, val_dataloader, criterion, metric, device=device)
evaluate_model_device(model, test_dataloader, criterion, metric, device=device)

## Few Impotant Python Additonal Helpers based on Multiclass Classification

```python

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchmetrics
from torchvision import transforms
import torchvision
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/iris.csv"
df = pd.read_csv(url, header=None)

In [ ]:
df.head()

In [ ]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

print(X.shape, y.shape)

In [ ]:
le = LabelEncoder().fit(y)
y = le.transform(y)

In [ ]:
# split into train, validation and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

#### Datasets and Dataloaders

In [ ]:
class IrisDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __getitem__(self, index):
        x = self.X[index]
        y = self.y[index]
        return x, y

    def __len__(self):
        return self.X.shape[0]

In [ ]:
train_ds = IrisDataset(X_train, y_train)
val_ds = IrisDataset(X_val, y_val)
test_ds = IrisDataset(X_test, y_test)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=32,
    shuffle=True,
)

val_loader = DataLoader(
    dataset=val_ds,
    batch_size=32,
    shuffle=False,
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=32,
    shuffle=False,
)

#### Iris Model

In [ ]:
class IrisModel(nn.Module):
    """
    Simple Feedforward Neural Network for Iris classification.
    """

    def __init__(self, in_features, hidden_units, out_features):
        super(IrisModel, self).__init__()
        self.layer_1 = nn.Linear(in_features, hidden_units)
        self.layer_2 = nn.Linear(hidden_units, 64)
        self.layer_3 = nn.Linear(64, out_features)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.relu(x)
        x = self.layer_3(x)
        return x

In [ ]:
torch.manual_seed(42)
model = IrisModel(in_features=4, hidden_units=8, out_features=3)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
device = torch.device("cpu")

In [ ]:
def training_loop_device(model, train_loader, criterion, optimizer, num_epochs, metric, device):
    """Training loop for the model with tqdm progress bar."""
    metric = metric.to(device)
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for batch_idx, (X_batch, y_batch) in enumerate(pbar):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            preds = model(X_batch)
            loss = criterion(preds, y_batch)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

            pbar.set_postfix(loss=loss.item())

        # calculate metrics
        accuracy = metric.compute()
        metric.reset()

        avg_loss = epoch_loss / len(train_loader) if len(train_loader) > 0 else 0.0
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")

In [ ]:
def evaluate_model_device(model, test_loader, criterion, metric, device):
    """Evaluate the model on the test set."""
    metric = metric.to(device)
    model = model.eval()
    test_loss = 0.0
    with torch.inference_mode():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            test_loss += loss.item()
            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

    accuracy = metric.compute()
    metric.reset()
    test_loss /= len(test_loader)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {accuracy:.4f}")

In [ ]:
training_loop_device(model, train_loader, criterion, optimizer, num_epochs=100, metric=metric, device=device)

In [ ]:
evaluate_model_device(model, train_loader, criterion, metric, device=device)
evaluate_model_device(model, val_loader, criterion, metric, device=device)
evaluate_model_device(model, test_loader, criterion, metric, device=device)

#### Model Checkpointing and Early Stopping

In [ ]:
def checkpoint(model, filename):
    torch.save(model.state_dict(), filename)


def resume(model, filename):
    model.load_state_dict(torch.load(filename))

In [ ]:
torch.manual_seed(42)
model = IrisModel(in_features=4, hidden_units=8, out_features=3)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
device = torch.device("cpu")

In [ ]:
training_loop_device(model, train_loader, criterion, optimizer, num_epochs=100, metric=metric, device=device)

In [ ]:
# saving the model
os.makedirs("model", exist_ok=True)
checkpoint(model, "model/iris_model.pth")

In [ ]:
# load the model
model2 = IrisModel(in_features=4, hidden_units=8, out_features=3)  # hollow model
resume(model2, "model/iris_model.pth")

In [ ]:
evaluate_model_device(model2, train_loader, criterion, metric, device=device)
evaluate_model_device(model2, val_loader, criterion, metric, device=device)
evaluate_model_device(model2, test_loader, criterion, metric, device=device)

In [ ]:
# If you wnat to resume training after a certain while then you have to save both model state_dict and optimizer state_dict
def checkpoint_full(model, optimizer, filename):
    checkpoint = {"model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict()}
    torch.save(checkpoint, filename)


def resume_full(model, optimizer, filename):
    checkpoint = torch.load(filename)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

In [ ]:
torch.manual_seed(42)
model = IrisModel(in_features=4, hidden_units=8, out_features=3)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
device = torch.device("cpu")

In [ ]:
training_loop_device(model, train_loader, criterion, optimizer, num_epochs=100, metric=metric, device=device)

In [ ]:
checkpoint_full(model, optimizer, "model/iris_model_full.pth")
# load the model and optimizer
model2 = IrisModel(in_features=4, hidden_units=8, out_features=3)  # hollow model
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)
resume_full(model2, optimizer2, "model/iris_model_full.pth")

In [ ]:
# train again after resuming
training_loop_device(model2, train_loader, criterion, optimizer2, num_epochs=100, metric=metric, device=device)

#### early stopping

In [ ]:
def training_loop_device_with_es(model, train_loader, criterion, optimizer, num_epochs, metric, device, patience):
    """Training loop for the model with tqdm progress bar and early stopping."""
    best_accuracy = -1
    metric = metric.to(device)
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for batch_idx, (X_batch, y_batch) in enumerate(pbar):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            preds = model(X_batch)
            loss = criterion(preds, y_batch)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

            pbar.set_postfix(loss=loss.item())

        # calculate metrics
        accuracy = metric.compute()
        metric.reset()

        avg_loss = epoch_loss / len(train_loader) if len(train_loader) > 0 else 0.0
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")

        if accuracy >= best_accuracy:
            best_accuracy = accuracy
            best_epoch = epoch
            checkpoint(model, "model/iris_model_best.pth")
            print(f"New best model saved with accuracy: {best_accuracy:.4f}")
        elif epoch - best_epoch >= patience:
            print("Early stopping triggered.")
            break

In [ ]:
# Reintialize model, optimizer, criterion, metric, device
torch.manual_seed(42)
model = IrisModel(in_features=4, hidden_units=8, out_features=3)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
device = torch.device("cpu")

In [ ]:
# retrain and now till 200 epochs with early stopping patience of 10 epochs
training_loop_device_with_es(
    model, train_loader, criterion, optimizer, num_epochs=200, metric=metric, device=device, patience=10
)

In [ ]:
evaluate_model_device(model, train_loader, criterion, metric, device=device)
evaluate_model_device(model, val_loader, criterion, metric, device=device)
evaluate_model_device(model, test_loader, criterion, metric, device=device)

#### VIZ models

In [ ]:
from torchview import draw_graph

model_graph = draw_graph(
    model, torch.zeros(1, 4), save_graph=True, filename="iris_model_graph", directory="./artifacts"
)
model_graph.visual_graph

In [ ]:
from torchinfo import summary

summary(model, input_size=(1, 4), verbose=0)

In [ ]:
from torchvista import trace_model

trace_model(model, torch.zeros(1, 4), export_format="png")

#### LR SCHEDULER

Simple learning rates scheduler like linear decay, step decay, exponential decay and custom schedulers can be implemented using torch.optim.lr_scheduler module.

In [ ]:
from torch.optim.lr_scheduler import StepLR, ExponentialLR, ReduceLROnPlateau, LinearLR, LambdaLR

In [ ]:
# step_lr_scheduler = StepLR(optimizer, step_size=30, gamma=0.1)
# exponential_lr_scheduler = ExponentialLR(optimizer, gamma=0.9)
# reduce_on_plateau_scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)
# linear_lr_scheduler = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=50)

# def lr_lambda(epoch):
#     # LR to be 0.1 * (1/1+0.01*epoch)
#     base_lr = 0.1
#     factor = 0.01
#     return base_lr/(1+factor*epoch)
# lambda_lr_scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

We will show one example of StepLR scheduler integrated into the training loop with early stopping.

In [101]:
def training_loop_device_with_es_lrs(
    model, train_loader, criterion, optimizer, num_epochs, metric, device, patience, lr_scheduler
):
    """Training loop for the model with tqdm progress bar and early stopping."""
    best_accuracy = -1
    metric = metric.to(device)
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for batch_idx, (X_batch, y_batch) in enumerate(pbar):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            preds = model(X_batch)
            loss = criterion(preds, y_batch)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

            pbar.set_postfix(loss=loss.item())

        before_lr = optimizer.param_groups[0]["lr"]
        lr_scheduler.step()
        after_lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch+1}: Learning Rate changed from {before_lr:.6f} to {after_lr:.6f}")

        # calculate metrics
        accuracy = metric.compute()
        metric.reset()

        avg_loss = epoch_loss / len(train_loader) if len(train_loader) > 0 else 0.0
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")

        if accuracy >= best_accuracy:
            best_accuracy = accuracy
            best_epoch = epoch
            checkpoint(model, "model/iris_model_best.pth")
            print(f"New best model saved with accuracy: {best_accuracy:.4f}")
        elif epoch - best_epoch >= patience:
            print("Early stopping triggered.")
            break

In [102]:
model = IrisModel(in_features=4, hidden_units=8, out_features=3)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
device = torch.device("cpu")
lr_scheduler = ExponentialLR(optimizer, gamma=0.99)

In [103]:
training_loop_device_with_es_lrs(
    model,
    train_loader,
    criterion,
    optimizer,
    num_epochs=200,
    metric=metric,
    device=device,
    patience=20,
    lr_scheduler=lr_scheduler,
)

Epoch 1/200:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 1: Learning Rate changed from 0.010000 to 0.009900
Epoch [1/200], Loss: 1.0690, Accuracy: 0.4722
New best model saved with accuracy: 0.4722


Epoch 2/200:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 2: Learning Rate changed from 0.009900 to 0.009801
Epoch [2/200], Loss: 0.8322, Accuracy: 0.7037
New best model saved with accuracy: 0.7037


Epoch 3/200:   0%|          | 0/4 [00:00<?, ?it/s]

...

Epoch 45: Learning Rate changed from 0.006426 to 0.006362
Epoch [45/200], Loss: 0.0820, Accuracy: 0.9722


Epoch 46/200:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 46: Learning Rate changed from 0.006362 to 0.006298
Epoch [46/200], Loss: 0.0654, Accuracy: 0.9630


Epoch 47/200:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 47: Learning Rate changed from 0.006298 to 0.006235
Epoch [47/200], Loss: 0.0786, Accuracy: 0.9630
Early stopping triggered.


For LR Reduce on Plateau, call lr_scheduler.step(val_loss) instead of lr_scheduler.step() and we need to calculate val_loss at the end of each epoch

In [104]:
class LRScheduler:
    """
    Learning rate scheduler. If the validation loss does not decrease for the
    given number of `patience` epochs, then the learning rate will decrease by
    by given `factor`.
    """

    def __init__(self, optimizer, patience=5, min_lr=1e-6, factor=0.5):
        """
        new_lr = old_lr * factor
        :param optimizer: the optimizer we are using
        :param patience: how many epochs to wait before updating the lr
        :param min_lr: least lr value to reduce to while updating
        :param factor: factor by which the lr should be updated
        """
        self.optimizer = optimizer
        self.patience = patience
        self.min_lr = min_lr
        self.factor = factor
        self.lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode="min",
            patience=self.patience,
            factor=self.factor,
            min_lr=self.min_lr,
        )

    def __call__(self, val_loss):
        self.lr_scheduler.step(val_loss)

In [105]:
model = IrisModel(in_features=4, hidden_units=8, out_features=3)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
device = torch.device("cpu")
lr_scheduler = LRScheduler(optimizer, patience=5)

In [106]:
# def training_loop_single_epoch(model, train_loader, criterion, optimizer, metric, device):
#     """Single epoch training loop for the model."""
#     model.train()
#     epoch_loss = 0.0
#     pbar = tqdm(train_loader, desc=f"Training", leave=False)
#     for batch_idx, (X_batch, y_batch) in enumerate(pbar):
#         X_batch = X_batch.to(device)
#         y_batch = y_batch.to(device)

#         # Forward pass
#         preds = model(X_batch)
#         loss = criterion(preds, y_batch)

#         # Backward pass and optimization
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         epoch_loss += loss.item()

#         pred_labels = preds.argmax(dim=1)
#         metric.update(pred_labels, y_batch)

#         pbar.set_postfix(loss=loss.item())

#     # calculate metrics
#     accuracy = metric.compute()
#     metric.reset()

#     avg_loss = epoch_loss / len(train_loader) if len(train_loader) > 0 else 0.0
#     return avg_loss, accuracy

In [107]:
# def validation_loop_single_epoch(model, val_loader, criterion, metric, device):
#     """Single epoch validation loop for the model."""
#     model.eval()
#     val_loss = 0.0
#     with torch.inference_mode():
#         for X_batch, y_batch in val_loader:
#             X_batch = X_batch.to(device)
#             y_batch = y_batch.to(device)
#             preds = model(X_batch)
#             loss = criterion(preds, y_batch)
#             val_loss += loss.item()
#             pred_labels = preds.argmax(dim=1)
#             metric.update(pred_labels, y_batch)

#     accuracy = metric.compute()
#     metric.reset()
#     val_loss /= len(val_loader)
#     return val_loss, accuracy

In [108]:
# def training_loop_device_with_es_lrs_custom(
#     model, train_loader, val_loader, criterion, optimizer, num_epochs, metric, device, patience, lr_scheduler
# ):
#     """Training loop for the model with tqdm progress bar and early stopping."""
#     best_accuracy = -1
#     metric = metric.to(device)
#     for epoch in range(num_epochs):
#         train_loss, train_accuracy = training_loop_single_epoch(
#             model, train_loader, criterion, optimizer, metric, device
#         )
#         val_loss, val_accuracy = validation_loop_single_epoch(model, val_loader, criterion, metric, device)

#         lr_scheduler(val_loss)

#         print(
#             f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}"
#         )

#         if val_accuracy >= best_accuracy:
#             best_accuracy = val_accuracy
#             best_epoch = epoch
#             checkpoint(model, "model/iris_model_best.pth")
#             print(f"New best model saved with accuracy: {best_accuracy:.4f}")
#         elif epoch - best_epoch >= patience:
#             print("Early stopping triggered.")
#             break

#### OPTIMIZED VERSION OF THE ABOVE TRAINING LOOP WITH LR SCHEDULER AND EARLY STOPPING ####

In [109]:
def training_loop_single_epoch(model, train_loader, criterion, optimizer, metric, device):
    """Single epoch training loop for the model."""
    from tqdm.auto import tqdm

    model.train()
    epoch_loss = 0.0
    metric = metric.to(device)
    pbar = tqdm(train_loader, desc=f"Training", leave=False)
    for batch_idx, (X_batch, y_batch) in enumerate(pbar):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        preds = model(X_batch)
        loss = criterion(preds, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        pred_labels = preds.argmax(dim=1)
        metric.update(pred_labels, y_batch)

        pbar.set_postfix(loss=loss.item())

    acc = metric.compute()
    metric.reset()
    acc = acc.item() if hasattr(acc, "item") else float(acc)

    avg_loss = epoch_loss / len(train_loader) if len(train_loader) > 0 else 0.0
    return avg_loss, acc

In [110]:
def validation_loop_single_epoch(model, val_loader, criterion, metric, device):
    """Single epoch validation loop for the model."""
    model.eval()
    val_loss = 0.0
    metric = metric.to(device)
    with torch.inference_mode():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            val_loss += loss.item()
            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

    acc = metric.compute()
    metric.reset()
    acc = acc.item() if hasattr(acc, "item") else float(acc)

    val_loss /= len(val_loader) if len(val_loader) > 0 else 1.0
    return val_loss, acc

In [111]:
def training_loop_device_with_es_lrs_custom(
    model, train_loader, val_loader, criterion, optimizer, num_epochs, metric, device, patience, lr_scheduler
):
    """Training loop for the model with tqdm progress bar and early stopping."""
    best_accuracy = -1.0
    metric = metric.to(device)
    best_epoch = 0
    for epoch in range(num_epochs):
        train_loss, train_accuracy = training_loop_single_epoch(
            model, train_loader, criterion, optimizer, metric, device
        )
        val_loss, val_accuracy = validation_loop_single_epoch(model, val_loader, criterion, metric, device)

        # ensure numeric
        lr_scheduler(float(val_loss))

        print(
            f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}"
        )

        if val_accuracy > best_accuracy:
            best_accuracy = float(val_accuracy)
            best_epoch = epoch
            checkpoint(model, "model/iris_model_best.pth")
            print(f"New best model saved with accuracy: {best_accuracy:.4f}")
        elif epoch - best_epoch >= patience:
            print("Early stopping triggered.")
            break

In [112]:
training_loop_device_with_es_lrs_custom(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    num_epochs=200,
    metric=metric,
    device=device,
    patience=20,
    lr_scheduler=lr_scheduler,
)

Training:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch [1/200], Train Loss: 1.0035, Train Accuracy: 0.4537, Val Loss: 0.8826, Val Accuracy: 0.5000
New best model saved with accuracy: 0.5000


Training:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch [2/200], Train Loss: 0.8581, Train Accuracy: 0.5463, Val Loss: 0.7624, Val Accuracy: 0.6667
New best model saved with accuracy: 0.6667


Training:   0%|          | 0/4 [00:00<?, ?it/s]

...

Epoch [24/200], Train Loss: 0.1062, Train Accuracy: 0.9630, Val Loss: 0.0313, Val Accuracy: 1.0000


Training:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch [25/200], Train Loss: 0.0983, Train Accuracy: 0.9630, Val Loss: 0.0315, Val Accuracy: 1.0000


Training:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch [26/200], Train Loss: 0.1022, Train Accuracy: 0.9537, Val Loss: 0.0298, Val Accuracy: 1.0000
Early stopping triggered.
